# Week 4 — Information Security in Distributed (Federated) Learning
### Trustworthy AI · Attacks & Defenses in Federated Learning (PyTorch demos)

Federated learning (FL) lets many clients jointly train a model **without sharing raw data** — only
model updates travel to the server. This improves privacy but opens **new attack surfaces**:
poisoned updates, fake clients, inference from gradients, and communication-layer threats.

This notebook builds a small **federated averaging (FedAvg)** system from scratch on MNIST and then
demonstrates the attacks and defenses from the Week 4 slides.

**Roadmap**
1. A FedAvg simulator (server + N clients, non-shared data)
2. *Attack:* Data poisoning by malicious clients (label-flip / backdoor)
3. *Attack:* Model poisoning (directly tampering with updates)
4. *Attack:* Sybil (many fake clients) & Free-rider
5. *Defense:* Byzantine-robust aggregation (Krum, Trimmed Mean, Median)
6. *Defense:* Anomaly / outlier detection on updates
7. *Attack:* Membership / property inference from gradients
8. *Defense:* Differential privacy + Secure aggregation (conceptual)
9. Governance & wrap-up

> **Note:** everything is runnable top-to-bottom (CPU is fine) but not executed for you.
> All code uses **PyTorch**.


## 0. Setup + FedAvg simulator
We split MNIST across N clients. Each round: clients train locally on their *private* shard, send updates, the server averages them.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import numpy as np
import copy

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", DEVICE)

tfm = transforms.ToTensor()
train_full = datasets.MNIST("./data", train=True,  download=True, transform=tfm)
test_full  = datasets.MNIST("./data", train=False, download=True, transform=tfm)

def to_tensors(ds, n):
    xs = torch.stack([ds[i][0] for i in range(n)])
    ys = torch.tensor([ds[i][1] for i in range(n)])
    return xs, ys

X, y = to_tensors(train_full, 6000)
Xte, yte = to_tensors(test_full, 2000)

N_CLIENTS = 10
# each client gets a disjoint shard -> data never leaves the client
shards = [(X[i::N_CLIENTS], y[i::N_CLIENTS]) for i in range(N_CLIENTS)]
print("clients:", N_CLIENTS, "| samples/client ~", len(shards[0][0]))


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.flatten(1)
        return self.fc2(F.relu(self.fc1(x)))

def get_flat(model):
    return torch.cat([p.data.flatten() for p in model.parameters()])

def set_flat(model, vec):
    i = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(vec[i:i+n].view_as(p)); i += n

def local_train(global_vec, data, epochs=1, lr=0.1):
    model = MLP().to(DEVICE); set_flat(model, global_vec.to(DEVICE))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    Xc, yc = data
    loader = DataLoader(TensorDataset(Xc, yc), batch_size=64, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    return get_flat(model).cpu()   # the update this client sends up

@torch.no_grad()
def test_acc(global_vec):
    model = MLP().to(DEVICE); set_flat(model, global_vec.to(DEVICE)); model.eval()
    pred = model(Xte.to(DEVICE)).argmax(1).cpu()
    return (pred == yte).float().mean().item()


In [ ]:
def fedavg(client_fn, aggregate=None, rounds=15, epochs=1):
    """client_fn(cid, global_vec) -> update vector. aggregate(list_of_updates)->vec."""
    if aggregate is None:
        aggregate = lambda ups: torch.stack(ups).mean(0)
    global_vec = get_flat(MLP())
    hist = []
    for r in range(rounds):
        updates = [client_fn(cid, global_vec) for cid in range(len(shards))]
        global_vec = aggregate(updates)
        acc = test_acc(global_vec)
        hist.append(acc)
    return global_vec, hist

# Honest baseline
def honest_client(cid, g): return local_train(g, shards[cid])
g_clean, hist_clean = fedavg(honest_client, rounds=15)
print("Final global accuracy (all honest):", round(hist_clean[-1], 4))


## 1. Data Poisoning by Malicious Clients
Malicious clients train on **poisoned local data**. Even a small fraction can degrade the global
model. We use label-flipping + a backdoor trigger, exactly as in the centralized case but injected
through client updates the server can't inspect directly.


In [ ]:
TARGET = 0
def add_trigger(x):
    x = x.clone(); x[:, -3:, -3:] = 1.0; return x

MAL = {0, 1, 2}   # 3 of 10 clients are malicious

def poison_shard(Xc, yc):
    Xc, yc = Xc.clone(), yc.clone()
    yc[yc == 7] = 1                          # label flip 7 -> 1
    k = int(0.4 * len(Xc))
    idx = torch.randperm(len(Xc))[:k]
    Xc[idx] = add_trigger(Xc[idx]); yc[idx] = TARGET   # backdoor
    return Xc, yc

def poisoning_client(cid, g):
    data = poison_shard(*shards[cid]) if cid in MAL else shards[cid]
    return local_train(g, data)

g_pois, hist_pois = fedavg(poisoning_client, rounds=15)

Xtrig = add_trigger(Xte)
@torch.no_grad()
def backdoor_asr(vec):
    m = MLP().to(DEVICE); set_flat(m, vec.to(DEVICE)); m.eval()
    return (m(Xtrig.to(DEVICE)).argmax(1).cpu() == TARGET).float().mean().item()

print("Clean-data accuracy (poisoned FL):", round(hist_pois[-1], 4))
print("Backdoor attack success rate:     ", round(backdoor_asr(g_pois), 4))


## 2. Model Poisoning Attacks
Stronger than data poisoning: the attacker doesn't bother with samples — it **directly manipulates
the update vector** before sending it (e.g. scales it to dominate the average). This can stealthily
insert backdoors or wreck accuracy.


In [ ]:
def model_poisoning_client(cid, g):
    if cid in MAL:
        # train on backdoored data, then BOOST the update to overpower honest ones
        upd = local_train(poison_shard(*shards[cid]), None) if False else local_train(g, poison_shard(*shards[cid]))
        boosted = g + (upd - g) * 5.0   # scale the malicious delta x5
        return boosted
    return local_train(g, shards[cid])

g_mp, hist_mp = fedavg(model_poisoning_client, rounds=15)
print("Accuracy under model poisoning:", round(hist_mp[-1], 4))
print("Backdoor ASR under model poisoning:", round(backdoor_asr(g_mp), 4), "<- boosting makes it worse")


## 3. Sybil & Free-Rider Attacks
- **Sybil:** one attacker registers many fake clients, so malicious updates dominate aggregation.
- **Free-rider:** a client sends fake/near-zero updates but still receives the improved global model,
  wasting resources and lowering fairness.


In [ ]:
# Sybil: turn the 3 malicious identities into 6 by cloning their poisoned updates
def sybil_client_updates(g):
    updates = []
    for cid in range(len(shards)):
        if cid in MAL:
            u = local_train(g, poison_shard(*shards[cid]))
            updates.append(u); updates.append(u.clone())   # each malicious id counts twice
        else:
            updates.append(local_train(g, shards[cid]))
    return updates

# Free-rider: sends the global vector back unchanged (no real training)
def free_rider_update(g): return g.clone()

# quick illustration of Sybil weight amplification on a single round
g0 = get_flat(MLP())
ups = sybil_client_updates(g0)
print(f"Honest identities: {len(shards)-len(MAL)}, malicious identities after Sybil: {len(ups)-(len(shards)-len(MAL))}")
print("=> malicious share of the average grew from",
      f"{len(MAL)/len(shards):.0%} to {(2*len(MAL))/(len(ups)):.0%}")


## 4. Defense — Byzantine-Robust Aggregation
Instead of a plain mean, use aggregators that **filter or downweight outliers**: Median, Trimmed
Mean, and **Krum** (pick the update closest to its nearest neighbors). These blunt poisoning and
Sybil influence.


In [ ]:
def agg_mean(ups):    return torch.stack(ups).mean(0)
def agg_median(ups):  return torch.stack(ups).median(0).values
def agg_trimmed(ups, beta=0.2):
    S = torch.stack(ups); k = int(beta * S.shape[0])
    S_sorted, _ = S.sort(0)
    return S_sorted[k:S.shape[0]-k].mean(0) if S.shape[0]-2*k > 0 else S.mean(0)
def agg_krum(ups, f=3):
    S = torch.stack(ups); n = S.shape[0]
    d = torch.cdist(S, S)                       # pairwise distances
    m = n - f - 2
    scores = []
    for i in range(n):
        nb, _ = d[i].sort()
        scores.append(nb[1:m+1].sum())          # sum of closest m neighbors
    return S[int(torch.tensor(scores).argmin())]

for name, agg in [("mean", agg_mean), ("median", agg_median),
                  ("trimmed", agg_trimmed), ("krum", agg_krum)]:
    g, h = fedavg(poisoning_client, aggregate=agg, rounds=15)
    print(f"{name:8s} | acc={h[-1]:.3f} | backdoor ASR={backdoor_asr(g):.3f}")


## 5. Defense — Anomaly / Outlier Detection on Updates
The server watches for **unusual update patterns** — e.g. abnormally large gradient norms often
signal poisoning or a boosted model-poisoning update. Flag and drop those clients.


In [ ]:
def anomaly_filter_agg(ups, z=2.0):
    S = torch.stack(ups)
    norms = S.norm(dim=1)
    mu, sd = norms.mean(), norms.std() + 1e-9
    keep = (norms - mu).abs() < z * sd          # drop far-from-mean norms
    kept = S[keep]
    return kept.mean(0) if len(kept) else S.mean(0)

g_af, h_af = fedavg(model_poisoning_client, aggregate=anomaly_filter_agg, rounds=15)
print("Model-poisoning + anomaly filter -> acc:", round(h_af[-1], 4),
      "| backdoor ASR:", round(backdoor_asr(g_af), 4))


## 6. Inference Attacks from Gradients
Even without raw data, an adversary inspecting updates can attempt **membership inference** (was this
sample used?) and **property inference** (a hidden attribute of a client's data). We illustrate a
simple membership signal: a client's update responds more strongly to samples it trained on.


In [ ]:
# Compare per-sample gradient norm from a fresh global model:
# samples a client trained on tend to produce smaller loss / different gradient response.
def sample_loss(vec, Xs, ys):
    m = MLP().to(DEVICE); set_flat(m, vec.to(DEVICE)); m.eval()
    with torch.no_grad():
        logits = m(Xs.to(DEVICE))
        losses = F.cross_entropy(logits, ys.to(DEVICE), reduction="none").cpu()
    return losses

# client 5 is honest; train a local model on its shard, then measure loss on
# (a) its own data (members) vs (b) other data (non-members)
own_vec = local_train(get_flat(MLP()), shards[5], epochs=3)
mem_loss = sample_loss(own_vec, *shards[5])
non_loss = sample_loss(own_vec, Xte[:len(shards[5][0])], yte[:len(shards[5][0])])
print(f"Mean loss on members: {mem_loss.mean():.3f} | non-members: {non_loss.mean():.3f}")
print("Lower member loss => the update leaks membership information.")


## 7. Defense — Differential Privacy in FL + Secure Aggregation
- **DP in FL:** each client clips and adds noise to its update before sending, defending against
  membership/property inference (at some utility cost).
- **Secure aggregation (conceptual):** clients mask updates with pairwise-cancelling random terms so
  the server only ever sees the **sum**, never any individual update — protecting privacy even if the
  server is compromised. Below we implement DP client updates and sketch masked aggregation.


In [ ]:
def dp_client(cid, g, clip=3.0, sigma=0.5):
    upd = local_train(g, shards[cid])
    delta = upd - g
    norm = delta.norm()
    delta = delta * min(1.0, clip / (norm + 1e-9))      # clip
    delta = delta + torch.randn_like(delta) * sigma * clip / len(shards[cid][0])  # noise
    return g + delta

g_dp, h_dp = fedavg(dp_client, rounds=15)
print("DP-FL global accuracy:", round(h_dp[-1], 4), "(privacy vs utility trade-off)")

# --- Secure aggregation sketch: pairwise masks cancel in the sum ---
def secure_aggregate(updates):
    n = len(updates); dim = updates[0].numel()
    masks = torch.zeros(n, dim)
    for i in range(n):
        for j in range(i+1, n):
            r = torch.randn(dim)                # shared pairwise secret
            masks[i] += r; masks[j] -= r        # cancels when summed
    masked = [u + masks[k] for k, u in enumerate(updates)]
    total = torch.stack(masked).sum(0)          # server sees only this
    return total / n

# sanity check: masked sum == plain mean (server learns nothing per-client)
demo = [torch.randn(5) for _ in range(4)]
print("secure-agg == plain mean?",
      torch.allclose(secure_aggregate(demo), torch.stack(demo).mean(0), atol=1e-4))


## 8. Blockchain & Access-Control Defenses (conceptual)
The slides also list:
- **Byzantine-robust aggregation** — done above (§4).
- **Blockchain-based FL:** an immutable, traceable ledger of client contributions deters tampering
  and free-riding, and supports decentralized trust without a single trusted server (still emerging;
  scalability challenges).
- **Access control & communication security:** strong client authentication, end-to-end encryption of
  updates in transit, and **rate-limiting** to prevent model extraction and man-in-the-middle attacks.

These are usually system/protocol-level and integrated via infrastructure rather than model code.


## 9. Continuous Monitoring, Governance & Wrap-up

| Threat (slides) | Defense demonstrated |
|---|---|
| Data poisoning | Robust aggregation §4, anomaly detection §5 |
| Model poisoning | Anomaly/norm filtering §5, Krum §4 |
| Sybil / free-rider | Robust aggregation §4, (blockchain/auth §8) |
| Membership / property inference | Differential privacy §7 |
| Server sees updates | Secure aggregation §7 |
| Communication / MITM / DoS | Encryption, auth, rate-limiting §8 |

**Takeaways**
- FL keeps data local but is **not immune** — attacks target updates, aggregation, and communication.
- No single defense is enough: combine **robust aggregation + DP + secure aggregation + monitoring**.
- Security in FL is **layered**: technical safeguards *and* governance, roles, compliance, and ongoing
  monitoring, because threats evolve over time.
